# Lab 04: server, OpenAI API and parallelism

Lab: `corso/laboratori/lab-04-01-server-e-webui, lab-04-02-client-openai, lab-04-03-parallelismo`

Run the cells in order (Runtime > Run all). It downloads up to 1 GB and takes a few minutes.


## 1. Setup

Download the official llama.cpp binaries (build `b11046`, the one used in the course) and the lab files.

In [ ]:
import os, subprocess
V = "b11046"
HOME = os.path.expanduser("~")
REL = f"{HOME}/llama-lab/llama-{V}"
if not os.path.exists(f"{REL}/llama"):
    subprocess.run(
        f"mkdir -p {HOME}/llama-lab && cd {HOME}/llama-lab && "
        f"curl -sLO https://github.com/ggml-org/llama.cpp/releases/download/{V}/llama-{V}-bin-ubuntu-x64.tar.gz && "
        f"tar -xzf llama-{V}-bin-ubuntu-x64.tar.gz", shell=True, check=True)
os.environ["PATH"] = REL + ":" + os.environ["PATH"]
T = min(len(os.sched_getaffinity(0)), 4)                      # threads = available cores (more than the physical cores slows things down)
os.environ["T"] = str(T)
!llama version

In [ ]:
BASE = "/content" if os.path.isdir("/content") else HOME
if not os.path.exists(f"{BASE}/corso"):
    !git clone -q --depth 1 https://github.com/JoeBosi/llama-cpp-course.git {BASE}/corso
LABS = f"{BASE}/corso/corso/laboratori"
os.environ["LABS"] = LABS

Download the sample model (785 MiB, Apache 2.0) into the Hugging Face cache.

In [ ]:
!llama download -hf ggml-org/Qwen3.5-0.8B-GGUF 2>&1 | tail -3
!llama cli -cl

## 2. Start the server (lab 04-01)

`avvia_server` launches `llama serve` in the background and waits until it is ready.

In [ ]:
import subprocess, time, urllib.request

def avvia_server(args, porta=8080, attesa=300):
    """Start `llama serve` in the background and wait for /health."""
    log = open("/tmp/llama-server.log", "w")
    p = subprocess.Popen(["llama", "serve", *args.split(), "--host", "127.0.0.1", "--port", str(porta)],
                         stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(attesa):
        try:
            if urllib.request.urlopen(f"http://127.0.0.1:{porta}/health", timeout=2).status == 200:
                return p
        except Exception:
            time.sleep(1)
    raise RuntimeError("the server does not answer: " + open("/tmp/llama-server.log").read()[-1500:])

def ferma_server(p):
    p.terminate(); p.wait()

In [ ]:
p = avvia_server("-hf ggml-org/Qwen3.5-0.8B-GGUF -c 2048 -np 1 -t %d --metrics -a qwen-0.8b" % T)
print("server pronto")

In [ ]:
%%bash
curl -s http://127.0.0.1:8080/health ; echo
curl -s http://127.0.0.1:8080/v1/models | python3 -m json.tool | head -12

### A chat request

The API is OpenAI-compatible: `POST /v1/chat/completions`.

In [ ]:
%%bash
curl -s http://127.0.0.1:8080/v1/chat/completions -H "Content-Type: application/json" -d '{
  "messages": [{"role": "user", "content": "Qual è la capitale d'\''Italia? Rispondi in una frase."}],
  "max_tokens": 40, "temperature": 0,
  "chat_template_kwargs": {"enable_thinking": false}}' | python3 -m json.tool | head -30

In [ ]:
%%bash
curl -s http://127.0.0.1:8080/tokenize -H "Content-Type: application/json" -d '{"content":"Ciao mondo!","with_pieces":true}'; echo
curl -s http://127.0.0.1:8080/apply-template -H "Content-Type: application/json" \
  -d '{"messages":[{"role":"system","content":"Sei un assistente."},{"role":"user","content":"Ciao"}]}'; echo

## 3. Python and JavaScript clients (lab 04-02)

In [ ]:
!pip -q install openai

In [ ]:
%%bash
cd $LABS/lab-04-02-client-openai
python3 chat.py
python3 chat.py --stream | tail -3
node --version
node chat.mjs

## 4. Parallelism (lab 04-03)

With `-np 1` requests queue up. Restart the server with 4 slots and compare the aggregate throughput.

In [ ]:
%%bash
cd $LABS/lab-04-03-parallelismo
python3 carico.py --url http://127.0.0.1:8080 --richieste 4 --parallele 4 --token 24

In [ ]:
ferma_server(p)
p = avvia_server("-hf ggml-org/Qwen3.5-0.8B-GGUF -c 2048 -np 4 -t %d" % T)
print("server con 4 slot pronto")

In [ ]:
%%bash
cd $LABS/lab-04-03-parallelismo
python3 carico.py --url http://127.0.0.1:8080 --richieste 4 --parallele 4 --token 24

In [ ]:
ferma_server(p)